# 因子绩效评估

通过IC、分层的方式，来评价单因子或多因子组合在市场回测中的表现。

In [ ]:
from pathlib import Path

from factool import DuckPQSource
from example.evaluate import BacktestParams, run, show

## 参数配置

### 数据基础

进行因子绩效评估需要首先准备：

1. 基础行情数据源
2. 因子原始数据

通过指定数据库的位置DATA_PATH，为因子绩效评估的回测分析提供基础数据。

### 因子路径

回测获取数据均以“因子路径”作为数据定位的工具。因子路径为由指定分隔符（一般为`/`）连接的表名和列名，例如`barra/size`，表示barra表中的size列。另外，使用因子路径读取数据时，可以采用`AS`来取别称。例如`barra/size AS bs`，读取出的数据将会被命名为`bs`。

需要注意的是，能够由因子路径的方式读取的数据表，需要为面板数据，即应该包含多个标的在某段时间上的多个指标的表。其中每一个指标，作为一列存储。

In [ ]:
DATA_PATH = Path("data")
factor_source = DuckPQSource(DATA_PATH)

### 参数配置

配置类名为BacktestParams，具有如下参数：

1. factor_paths: 单因子回测时，提供一个因子路径；多因子回测时，提供一个字符串列表
2. begin: 回测起始日
3. end: 回测截止日
4. target_path: 用于回测的价格数据，默认使用因子路径`quotes_day/open_post AS open`
5. horizon: 作用于回测价格数据，表示回测计算未来收益时使用的周期长度，默认为5日
6. baseline_factors: 基准多因子模型，默认None，表示无基准多因子模型
7. min_list_days: 可纳入因子评估股票池的个股最小上市事件，单位为日，默认为90日
8. weight_path: 在加权组合时个股的权重，默认None，表示等权
9. ic_method: 计算IC使用的相关系数方法，可选"pearson"表示RankIC, "spearman"表示CorrIC，默认"spearman"
10. ic_roll_window: 表示计算滚动IC时使用的窗口长度，默认为60个交易日
11. ic_acf_lags: 表示计算IC自相关的滞后期，默认为10个交易日
12. ic_break_k: 计算IC是否发生显著变化的拐点数量，默认为1个
13. n_groups: 分层回测时，使用的分组数，默认10
14. monotonicity_use_excess: 计算分层回测收益的单调性时，是否去除组间均值，默认不剔除False
15. cs_add_intercept: 当设置了基准模型后，该参数生效，表示被评估因子相对基准模型回归时是否加入截距项，默认为True
16. cs_cov_type: 当设置了基准模型后，该参数生效，表示被评估因子相对基准模型回归时计算方差的类型，默认为"white"，表示经过White调整
17. cs_white_type: 当设置了基准模型、cs_cov_type为white时生效，决定了调整使用的方法，默认为"HC1"

In [ ]:
params = [
    BacktestParams(
        factor_paths="barra_size/mcap_float_a",  # TODO: 待确认因子路径
        begin="2015-01-01",
        end="2025-12-31",
    ),
    # 你可以添加更多测试组
]

## 启动回测

设定好一组参数实例，你可以使用`run_all`一键全部启动。但可能面临内存问题，根据内存大小来确定一组回测参数格式，一般一个参数需要占用1GB左右。

In [ ]:
results = run(factor_source, params)

In [ ]:
show(results)